# 02. OpenAI 호환 API와 코딩 평가

목표: Ornith server에 보낼 request를 만들고, 결과를 설명의 유창함이 아니라 test contract로 평가한다. 기본값 `DRY_RUN = True`는 network request를 보내지 않는다.

## 1. 모호한 요청을 검증 가능한 요청으로 바꾸기

빈 list의 평균을 어떻게 정의할지는 product contract다. 모델이 임의로 정하지 않도록 요구사항, 허용 변경 범위, 필수 test를 prompt에 넣는다.

In [ ]:
def build_bugfix_prompt() -> str:
    return """다음 Python 함수의 버그를 수정하라.

def average(numbers):
    return sum(numbers) / len(numbers)

계약:
- 빈 iterable이면 ValueError를 발생시킨다.
- 기존의 정상 숫자 입력 동작은 유지한다.
- 함수 signature는 변경하지 않는다.
- 수정된 code와 pytest test 3개를 제시한다.
- 변경 이유를 3문장 이내로 설명한다.
"""


prompt = build_bugfix_prompt()
print(prompt)
assert "ValueError" in prompt and "pytest" in prompt

## 2. request payload 만들기

정밀 coding task에 대해 공식 모델 카드가 권장한 `temperature=0.6`, `top_p=0.95`를 사용한다. 재현 실험에서는 모든 sampling parameter와 model revision을 기록한다.

In [ ]:
import json


def make_payload(prompt: str) -> dict:
    return {
        "model": "Ornith-1.5-9B",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.6,
        "top_p": 0.95,
        "max_tokens": 1024,
    }


payload = make_payload(prompt)
print(json.dumps(payload, ensure_ascii=False, indent=2))
assert payload["model"] == "Ornith-1.5-9B"

## 3. 선택적으로 local server 호출

아래 cell은 `DRY_RUN`을 직접 `False`로 바꾼 경우에만 localhost로 요청한다. 외부 server 주소나 API key를 notebook에 저장하지 않는다.

In [ ]:
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

DRY_RUN = True
LOCAL_ENDPOINT = "http://127.0.0.1:8000/v1/chat/completions"


def call_local_server(payload: dict, timeout_seconds: int = 60) -> dict:
    body = json.dumps(payload).encode("utf-8")
    request = Request(
        LOCAL_ENDPOINT,
        data=body,
        headers={"Content-Type": "application/json", "Authorization": "Bearer EMPTY"},
        method="POST",
    )
    with urlopen(request, timeout=timeout_seconds) as response:
        return json.loads(response.read().decode("utf-8"))


if DRY_RUN:
    result = {"status": "dry-run", "endpoint": LOCAL_ENDPOINT}
else:
    try:
        result = call_local_server(payload)
    except (HTTPError, URLError, TimeoutError) as error:
        result = {"status": "error", "type": type(error).__name__, "message": str(error)}

print(result)
assert DRY_RUN or "choices" in result or result.get("status") == "error"

## 4. 평가 rubric

응답을 자동 실행하기 전에 code block을 review하고 격리된 임시 환경에서 test한다. 점수 예시:

- 2점: 빈 iterable에서 명시한 `ValueError`가 발생한다.
- 2점: 정상 입력 결과가 유지된다.
- 2점: 필수 test 3개가 독립적으로 실행된다.
- 2점: 불필요한 dependency나 signature 변경이 없다.
- 2점: 설명과 실제 code가 일치한다.

같은 과제를 한국어와 영어로 실행하고 pass rate, latency, output token 수를 함께 기록하면 한국어 성능을 더 정직하게 비교할 수 있다.